# Environment setup

To reproduce this analysis:

1. Clone the repository
2. Create the conda environment:

conda env create -f environment.yml

3. Activate it:

conda activate saam_project

4. Open this notebook and run all cells from top to bottom.

# SAAM Project – Part I  
## 1. Data cleaning  

The datasets used in this notebook are cleaned versions of the original Datastream outputs.

The cleaning procedure follows the project guidelines:

- firms with completely missing rows in key datasets are removed  
- prices below 0.5 are treated as missing observations  
- missing prices at the beginning of the sample reflect firms not yet listed  
- missing prices at the end of the sample reflect delisting events  
- missing values are preserved in the return index dataset and handled later when constructing yearly investment sets  

## 2.1 Investment set construction  

This section constructs the investment universe used for the out-of-sample portfolio allocation.

Starting from the cleaned monthly return index data, we:

- compute monthly simple return series  
- define yearly investment sets based on data availability and carbon information  
- estimate rolling expected returns and covariance matrices using a 10-year historical window  

These quantities represent the necessary inputs for the portfolio optimization step described in Section 2.2.

### Step 1 — Load cleaned datasets

We load the cleaned monthly return index dataset, carbon emissions data, and static firm characteristics.

These datasets provide the inputs required to construct monthly return series and define the investment universe.

### Step 2 — Construct monthly simple returns

Monthly simple returns are computed from the return index series.

These return series will later be used to estimate:
- expected returns
- covariance matrices
- yearly investment sets based on data availability

In [5]:
import pandas as pd
import numpy as np

path = "../data/cleaned/"

ri_m = pd.read_csv(path + "RI_M_cleaned.csv", sep=";", na_values="N/A")
co2 = pd.read_csv(path + "CO2_S1_cleaned.csv", sep=";", na_values="N/A")
static = pd.read_csv(path + "STATIC_cleaned.csv", sep=";", na_values="N/A")

print("RI_M shape:", ri_m.shape)
print("CO2 shape:", co2.shape)
print("STATIC shape:", static.shape)

ri_m.head()

RI_M shape: (618, 316)
CO2 shape: (618, 29)
STATIC shape: (618, 4)


,NAME,ISIN,36525,36556,36585,36616,36644,36677,36707,36738,...,45777,45807,45838,45869,45898,45930,45961,45989,46022,46052
0,STRABAG SE,AT000000STR1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,"430,27","430,83","479,46","477,35","465,06","461,5","396,65","452,87","481,49","526,88"
1,FLUGHAFEN WIEN,AT00000VIE62,"147,79","156,25","153,83","158,62","137,27","148,95","159,68","151,08",...,"2339,19","2371,03","2478,96","2417,14","2434,84","2425,66","2409,99","2533,4","2591,68","2587,67"
2,RAIFFEISEN BANK INTL.,AT0000606306,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,"113,07","129,28","128,7","123,36","140,73","146,05","158,13","171,78","190,56","215,11"
3,ERSTE GROUP BANK,AT0000652011,"102,94","94,91","97,74","100,5","96,13","97,86","102,06","102,43",...,"1237,75","1536,73","1621,45","1761,42","1818,07","1867,75","1979,03","2087,48","2308,86","2488,84"
4,TELEKOM AUSTRIA,AT0000720008,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,"343,5","359,25","383,12","360,02","370,54","361,64","360,3","350,11","357,49","360,93"


### Step 3 — Prepare the monthly return index dataset

The monthly return index dataset is stored in wide format.

The first columns identify firms (`NAME`, `ISIN`), while the remaining columns correspond to monthly observations encoded as Excel serial dates.

Before computing returns, we:

- convert Excel serial dates into proper monthly timestamps
- convert return index values into numeric format

In [6]:
date_cols = ri_m.columns[2:]
date_nums = pd.to_numeric(date_cols, errors="raise")
dates = pd.to_datetime(date_nums, unit="D", origin="1899-12-30")

ri_m_wide = ri_m.copy()
ri_m_wide.columns = ["NAME", "ISIN"] + list(dates)

ri_m_wide = ri_m_wide.set_index(["ISIN", "NAME"])
ri_m_wide = ri_m_wide.sort_index(axis=1)

# Convert comma decimal values to numeric
ri_m_wide = ri_m_wide.apply(
    lambda col: pd.to_numeric(
        col.astype(str).str.replace(",", ".", regex=False),
        errors="coerce"
    )
)

print("Prepared RI matrix:", ri_m_wide.shape)
ri_m_wide.iloc[:, :5]

Prepared RI matrix: (618, 314)


,,1999-12-31 00:00:00,2000-01-31 00:00:00,2000-02-29 00:00:00,2000-03-31 00:00:00,2000-04-28 00:00:00
ISIN,NAME,,,,,
AT000000STR1,STRABAG SE,NaN,NaN,NaN,NaN,NaN
AT00000VIE62,FLUGHAFEN WIEN,147.79,156.25,153.83,158.62,137.27
AT0000606306,RAIFFEISEN BANK INTL.,NaN,NaN,NaN,NaN,NaN
AT0000652011,ERSTE GROUP BANK,102.94,94.91,97.74,100.50,96.13
AT0000720008,TELEKOM AUSTRIA,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...
SE0020050417,BOLIDEN ORD SHS,16.00,15.72,11.76,11.49,8.90
US0528001094,AUTOLIV,85.34,77.68,79.83,87.89,82.58
US70211M1099,PARTNER COMMS.ADR 1:1 DEAD - DELIST.22/09/23,159.82,113.50,105.00,102.69,66.01


### Step 4 — Construct monthly simple returns

Monthly simple returns are computed from the monthly return index series using:

\[
R_t = \frac{RI_t}{RI_{t-1}} - 1
\]

These return series will be used to estimate expected returns and covariance matrices in a 10-year rolling window, as required by the project instructions.

In [7]:
returns_wide = ri_m_wide.pct_change(axis=1)

print("Returns matrix shape:", returns_wide.shape)
returns_wide.iloc[:, :5]

Returns matrix shape: (618, 314)


,,1999-12-31 00:00:00,2000-01-31 00:00:00,2000-02-29 00:00:00,2000-03-31 00:00:00,2000-04-28 00:00:00
ISIN,NAME,,,,,
AT000000STR1,STRABAG SE,NaN,NaN,NaN,NaN,NaN
AT00000VIE62,FLUGHAFEN WIEN,NaN,0.057243,-0.015488,0.031138,-0.134598
AT0000606306,RAIFFEISEN BANK INTL.,NaN,NaN,NaN,NaN,NaN
AT0000652011,ERSTE GROUP BANK,NaN,-0.078007,0.029818,0.028238,-0.043483
AT0000720008,TELEKOM AUSTRIA,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...
SE0020050417,BOLIDEN ORD SHS,NaN,-0.017500,-0.251908,-0.022959,-0.225413
US0528001094,AUTOLIV,NaN,-0.089759,0.027678,0.100965,-0.060416
US70211M1099,PARTNER COMMS.ADR 1:1 DEAD - DELIST.22/09/23,NaN,-0.289826,-0.074890,-0.022000,-0.357192


In [8]:
print("Total missing values in returns:", returns_wide.isna().sum().sum())
print("First return column entirely missing:", returns_wide.iloc[:, 0].isna().all())

Total missing values in returns: 8847
First return column entirely missing: True


### Step 5 — Construct yearly investment sets

At the end of each year \(Y\), we define the investment set for year \(Y+1\).

Following the project instructions, a firm is included in the investment set if:

- it belongs to the assigned region
- it has a sufficient number of monthly return observations over the previous 10 years
- it has carbon data available at the end of year \(Y\)

Since the static dataset is already filtered for Europe, the main filters applied here are:

- availability of at least 36 monthly return observations in the 120-month rolling window
- availability of Scope 1 carbon data in year \(Y\)

In [13]:
co2_year_cols = co2.columns[2:]
co2_years = pd.to_numeric(co2_year_cols, errors="raise").astype(int)

co2_wide = co2.copy()
co2_wide.columns = ["NAME", "ISIN"] + list(co2_years)
co2_wide = co2_wide.set_index(["ISIN", "NAME"])

# Convert comma decimals to numeric
co2_wide = co2_wide.apply(
    lambda col: pd.to_numeric(
        col.astype(str).str.replace(",", ".", regex=False),
        errors="coerce"
    )
)

print("Prepared CO2 matrix:", co2_wide.shape)

december_dates = [d for d in returns_wide.columns if d.month == 12]
years_available = [d.year for d in december_dates]

print("December years available:", years_available[:5], "...", years_available[-5:])

investment_sets = {}

for dec_date in december_dates:
    # Require a full 10-year rolling window (120 months)
    window_start = dec_date - pd.DateOffset(months=119)

    if window_start < returns_wide.columns[0]:
        continue

    Y = dec_date.year

    # Rolling monthly return window up to end of year Y
    window_returns = returns_wide.loc[:, window_start:dec_date]

    # Filter 1: sufficient return history (at least 36 valid monthly observations)
    valid_return_counts = window_returns.notna().sum(axis=1)
    sufficient_history = valid_return_counts >= 36

    # Filter 2: carbon data available at end of year Y
    if Y in co2_wide.columns:
        carbon_available = co2_wide[Y].notna()
    else:
        carbon_available = pd.Series(False, index=co2_wide.index)

    # Common firms between returns and CO2 datasets
    common_index = returns_wide.index.intersection(co2_wide.index)

    eligible = common_index[
        sufficient_history.loc[common_index] &
        carbon_available.loc[common_index]
    ]

    investment_sets[Y] = list(eligible)

investment_set_sizes = pd.Series({year: len(firms) for year, firms in investment_sets.items()})

print("Number of yearly investment sets:", len(investment_sets))
print("First investable year:", investment_set_sizes.index[0])
print("Size first set:", investment_set_sizes.iloc[0])
print("Last investable year:", investment_set_sizes.index[-1])
print("Size last set:", investment_set_sizes.iloc[-1])

investment_set_sizes

Prepared CO2 matrix: (618, 27)
December years available: [1999, 2000, 2001, 2002, 2003] ... [2021, 2022, 2023, 2024, 2025]
Number of yearly investment sets: 17
First investable year: 2009
Size first set: 362
Last investable year: 2025
Size last set: 614


2009    362
2010    404
2011    436
2012    453
2013    477
2014    495
2015    511
2016    518
2017    532
2018    551
2019    577
2020    597
2021    605
2022    613
2023    613
2024    613
2025    614
dtype: int64

In [14]:
first_year = investment_set_sizes.index[0]
first_set = investment_sets[first_year]

print("First investable year:", first_year)
print("Number of firms:", len(first_set))
first_set[:10]

First investable year: 2009
Number of firms: 362


[('AT0000652011', 'ERSTE GROUP BANK'),
 ('AT0000720008', 'TELEKOM AUSTRIA'),
 ('AT0000743059', 'OMV'),
 ('AT0000746409', 'VERBUND'),
 ('BE0003470755', 'SOLVAY'),
 ('BE0003565737', 'KBC GROUP'),
 ('BE0003735496', 'ORANGE BELGIUM'),
 ('BE0003739530', 'UCB'),
 ('BE0003810273', 'PROXIMUS'),
 ('BE0974258874', 'BEKAERT (D)')]

### Step 6 — Compute rolling expected returns and covariance matrices

For each investable year \(Y\), we estimate the inputs required for the minimum-variance optimization.

Following the project instructions, we use a 10-year rolling window (120 monthly observations) ending in December of year \(Y\) to compute:

- the vector of expected returns \(\hat{\mu}_Y\)
- the covariance matrix \(\Sigma_Y\)

These quantities are estimated only on the firms belonging to the investment set of year \(Y\), so that the next step can solve the long-only minimum-variance allocation problem out of sample.

In [15]:
mu_by_year = {}
cov_by_year = {}
returns_window_by_year = {}

for Y, firms in investment_sets.items():
    # December date associated with year Y
    dec_date = pd.Timestamp(f"{Y}-12-31")
    
    # Rolling 10-year window = 120 months up to end of year Y
    window_start = dec_date - pd.DateOffset(months=119)
    window_returns = returns_wide.loc[firms, window_start:dec_date].copy()
    
    # Store returns window for later use in portfolio construction
    returns_window_by_year[Y] = window_returns
    
    # Expected returns: average over available monthly returns
    mu_y = window_returns.mean(axis=1)
    
    # Covariance matrix
    cov_y = window_returns.T.cov()
    
    mu_by_year[Y] = mu_y
    cov_by_year[Y] = cov_y

print("Number of expected return vectors:", len(mu_by_year))
print("Number of covariance matrices:", len(cov_by_year))

first_year = list(mu_by_year.keys())[0]
print("First year:", first_year)
print("Mu vector size:", mu_by_year[first_year].shape)
print("Cov matrix shape:", cov_by_year[first_year].shape)

Number of expected return vectors: 17
Number of covariance matrices: 17
First year: 2009
Mu vector size: (362,)
Cov matrix shape: (362, 362)


In [16]:
first_year = list(mu_by_year.keys())[0]

print("First investable year:", first_year)
print("\nExpected returns (first 10 firms):")
display(mu_by_year[first_year].head(10))

print("\nCovariance matrix (top-left 5x5 block):")
display(cov_by_year[first_year].iloc[:5, :5])

First investable year: 2009

Expected returns (first 10 firms):


ISIN          NAME            
AT0000652011  ERSTE GROUP BANK    0.020325
AT0000720008  TELEKOM AUSTRIA     0.012606
AT0000743059  OMV                 0.020566
AT0000746409  VERBUND             0.015337
BE0003470755  SOLVAY              0.007461
BE0003565737  KBC GROUP           0.010113
BE0003735496  ORANGE BELGIUM      0.008748
BE0003739530  UCB                 0.006128
BE0003810273  PROXIMUS            0.007370
BE0974258874  BEKAERT (D)         0.016331
dtype: float64


Covariance matrix (top-left 5x5 block):


,ISIN,AT0000652011,AT0000720008,AT0000743059,AT0000746409,BE0003470755
,NAME,ERSTE GROUP BANK,TELEKOM AUSTRIA,OMV,VERBUND,SOLVAY
ISIN,NAME,,,,,
AT0000652011,ERSTE GROUP BANK,0.016025,0.004052,0.005420,0.006875,0.006509
AT0000720008,TELEKOM AUSTRIA,0.004052,0.007710,0.003031,0.002705,0.001985
AT0000743059,OMV,0.005420,0.003031,0.010722,0.005318,0.002764
AT0000746409,VERBUND,0.006875,0.002705,0.005318,0.009061,0.003710
BE0003470755,SOLVAY,0.006509,0.001985,0.002764,0.003710,0.006000


In [17]:
for Y in cov_by_year:
    cov_y = cov_by_year[Y]
    assert cov_y.shape[0] == cov_y.shape[1], f"Covariance matrix not square in year {Y}"
    assert cov_y.index.equals(cov_y.columns), f"Covariance matrix labels mismatch in year {Y}"

print("All covariance matrices are square and properly labelled.")

All covariance matrices are square and properly labelled.


## 2.2 Minimum-variance portfolio allocation